# Toilav Bot — Chatbot Experimentation Notebook

Simula mensajes de WhatsApp entrantes directamente contra el servicio de chatbot. Cubre todos los casos de uso del roadmap.

## Setup rápido
```bash
cd app/services/chatbot
uv run python main.py   # levanta el servidor en :8001
```

## Sections
- **A** Health check
- **B** Webhook verification (GET)
- **C** Seguridad / firma HMAC (POST)
- **D** Unit tests sin servidor
- **E** Flujo de cliente vía HTTP
- **F** Comandos del dueño
- **G** Deduplicación de mensajes
- **H** Batching / debounce
- **I** Usuarios concurrentes
- **J** Edge cases
- **K** E2E completo
- **L** Tests directos del agente LLM

> **Nota sobre respuestas:** el webhook siempre devuelve `{"status": "ok"}` inmediatamente. La respuesta real del bot se envía al WhatsApp API en un background task. Con tokens de prueba el envío falla silenciosamente, pero el **log del servidor** muestra la respuesta generada. Usa `uv run python main.py` con logging visible.

---
## 0. Setup

Ejecuta estas celdas **siempre primero**. Las variables de entorno deben coincidir con las del servicio en ejecución (`.env` o las por defecto del `conftest.py`).

In [ ]:
import os

# Ajusta estos valores para que coincidan con tu .env / config.py
os.environ.setdefault('WHATSAPP_ACCESS_TOKEN', 'test-access-token')
os.environ.setdefault('APP_ID', 'test-app-id')
os.environ.setdefault('APP_SECRET', '0374f2ecf3b60940e7375e6fe76b0510')   # DEBE coincidir con el servicio
os.environ.setdefault('WHATSAPP_API_VERSION', 'v18.0')
os.environ.setdefault('PHONE_NUMBER_ID', '123456789')
os.environ.setdefault('VERIFY_TOKEN', '3A2obGyv2fDnkx8xCJw35vBd9mO_7vb5ozsBrJVUwfggA9iH6')
os.environ.setdefault('OWNER_WA_ID', '5215512345678')
os.environ.setdefault('OPENAI_API_KEY', 'sk-...')         # Necesario para Sección L

# URL del servicio
BASE_URL    = 'http://localhost:8001'
APP_SECRET  = os.environ['APP_SECRET']
VERIFY_TOKEN = os.environ['VERIFY_TOKEN']
OWNER_WA_ID = os.environ['OWNER_WA_ID']

print(f'BASE_URL:    {BASE_URL}')
print(f'OWNER_WA_ID: {OWNER_WA_ID}')
print(f'APP_SECRET:  {"*" * len(APP_SECRET)}')

In [12]:
import hashlib
import hmac as _hmac
import json
import time
import asyncio

import httpx

# Cliente HTTP síncrono para la mayoría de tests
client = httpx.Client(base_url=BASE_URL, timeout=30)

print('Imports OK')
print(f'httpx version: {httpx.__version__}')

Imports OK
httpx version: 0.28.1


In [13]:
# ── Generador de IDs únicos ──────────────────────────────────────────────
_msg_counter = 0

def next_wamid() -> str:
    global _msg_counter
    _msg_counter += 1
    return f'wamid.test{_msg_counter:06d}'

# ── Firma HMAC (replica security.py) ────────────────────────────────────
def sign(payload_bytes: bytes, secret: str = APP_SECRET) -> str:
    """Genera el header X-Hub-Signature-256 para un payload dado."""
    h = _hmac.new(secret.encode('utf-8'), msg=payload_bytes, digestmod=hashlib.sha256)
    return f'sha256={h.hexdigest()}'

def post_webhook(payload: dict, bad_secret: str | None = None) -> httpx.Response:
    """Serializa, firma y postea un payload a /webhook."""
    body = json.dumps(payload, ensure_ascii=False).encode('utf-8')
    sig  = sign(body, bad_secret or APP_SECRET)
    return client.post(
        '/webhook',
        content=body,
        headers={'Content-Type': 'application/json', 'X-Hub-Signature-256': sig},
    )

# ── Pretty print ──────────────────────────────────────────────────────────
def pprint(resp: httpx.Response) -> None:
    print(f'HTTP {resp.status_code}')
    try:
        print(json.dumps(resp.json(), indent=2, ensure_ascii=False))
    except Exception:
        print(repr(resp.text))

print('Helpers OK')

Helpers OK


In [14]:
# ── Constructores de payloads WhatsApp ──────────────────────────────────

def make_text_msg(
    wa_id: str,
    name: str,
    text: str,
    *,
    msg_id: str | None = None,
    timestamp: int | None = None,
) -> dict:
    return {
        'object': 'whatsapp_business_account',
        'entry': [{
            'id': 'entry_001',
            'changes': [{
                'field': 'messages',
                'value': {
                    'messaging_product': 'whatsapp',
                    'metadata': {'display_phone_number': '5215500000000', 'phone_number_id': '123456789'},
                    'contacts': [{'profile': {'name': name}, 'wa_id': wa_id}],
                    'messages': [{
                        'from': wa_id,
                        'id': msg_id or next_wamid(),
                        'timestamp': str(timestamp or int(time.time())),
                        'type': 'text',
                        'text': {'body': text},
                    }],
                },
            }],
        }],
    }

def make_image_msg(
    wa_id: str,
    name: str,
    *,
    caption: str | None = None,
    img_id: str = 'img_001',
    msg_id: str | None = None,
) -> dict:
    image: dict = {'id': img_id, 'mime_type': 'image/jpeg', 'sha256': 'abc123'}
    if caption:
        image['caption'] = caption
    return {
        'object': 'whatsapp_business_account',
        'entry': [{'changes': [{'value': {
            'contacts': [{'profile': {'name': name}, 'wa_id': wa_id}],
            'messages': [{
                'from': wa_id,
                'id': msg_id or next_wamid(),
                'timestamp': str(int(time.time())),
                'type': 'image',
                'image': image,
            }],
        }}]}],
    }

def make_typed_msg(wa_id: str, name: str, msg_type: str, *, msg_id: str | None = None) -> dict:
    """Mensaje de tipo no-texto (audio, sticker, reaction, etc.)."""
    return {
        'object': 'whatsapp_business_account',
        'entry': [{'changes': [{'value': {
            'contacts': [{'profile': {'name': name}, 'wa_id': wa_id}],
            'messages': [{
                'from': wa_id,
                'id': msg_id or next_wamid(),
                'timestamp': str(int(time.time())),
                'type': msg_type,
                msg_type: {'id': 'media_001'},
            }],
        }}]}],
    }

def make_interactive_msg(
    wa_id: str,
    name: str,
    button_title: str,
    button_id: str = 'btn_001',
) -> dict:
    """Respuesta a botón interactivo (list_reply o button_reply)."""
    return {
        'object': 'whatsapp_business_account',
        'entry': [{'changes': [{'value': {
            'contacts': [{'profile': {'name': name}, 'wa_id': wa_id}],
            'messages': [{
                'from': wa_id,
                'id': next_wamid(),
                'timestamp': str(int(time.time())),
                'type': 'interactive',
                'interactive': {'button_reply': {'id': button_id, 'title': button_title}},
            }],
        }}]}],
    }

def make_status_update(recipient_id: str, status: str = 'delivered') -> dict:
    """Status update (delivered / read). No es un mensaje, debe ignorarse."""
    return {
        'object': 'whatsapp_business_account',
        'entry': [{'changes': [{'value': {
            'messaging_product': 'whatsapp',
            'statuses': [{
                'id': next_wamid(),
                'status': status,
                'timestamp': str(int(time.time())),
                'recipient_id': recipient_id,
            }],
        }}]}],
    }

print('Payload builders OK')

Payload builders OK


In [5]:
def send_customer(wa_id: str, name: str, text: str, wait: float = 5.0) -> httpx.Response:
    """Envía mensaje de cliente y espera el background task (debounce 3s + LLM)."""
    payload = make_text_msg(wa_id, name, text)
    resp = post_webhook(payload)
    print(f'[{name}] {text!r:60s}  → HTTP {resp.status_code}')
    if wait > 0:
        time.sleep(wait)
    return resp

def send_owner(cmd: str, wait: float = 3.0) -> httpx.Response:
    """Envía un comando slash del dueño."""
    payload = make_text_msg(OWNER_WA_ID, 'Owner', cmd)
    resp = post_webhook(payload)
    print(f'[Owner] {cmd!r:60s}  → HTTP {resp.status_code}')
    if wait > 0:
        time.sleep(wait)
    return resp

print('Conversation helpers OK')
print('TIP: revisa los logs del servidor para ver las respuestas del LLM')

Conversation helpers OK
TIP: revisa los logs del servidor para ver las respuestas del LLM


In [15]:
import psycopg2

connection = psycopg2.connect(
    host='localhost',
    database='tremenda-test',
    user='admin',
    password='password'
)
cursor = connection.cursor()

ModuleNotFoundError: No module named 'psycopg2'

In [8]:
cursor.close()
connection.close()

In [16]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Customers:
    c_id: int
    c_whatsapp_id: str
    c_name: str

In [12]:
cursor.execute("SELECT c_id, c_whatsapp_id, c_name FROM customers")
row = cursor.fetchone()
row

(1, '5214731710428', 'Kevin')

In [ ]:
Customers(*row)

1

---
## A. Health Check

Verifica que el servicio esté levantado.

In [16]:
try:
    resp = client.get('/health')
    pprint(resp)
    assert resp.status_code == 200
    print('\n✓ Servicio activo')
except httpx.ConnectError:
    print('✗ Servicio no disponible — levántalo con: uv run python main.py')

HTTP 200
{
  "status": "ok"
}

✓ Servicio activo


---
## B. Verificación del Webhook (GET /webhook)

WhatsApp llama a `GET /webhook` al configurar la integración. Debe responder con el `hub.challenge` si el token es correcto.

In [19]:
# B.1: Token válido → 200 + echo del challenge
resp = client.get('/webhook', params={
    'hub.mode': 'subscribe',
    'hub.verify_token': VERIFY_TOKEN,
    'hub.challenge': 'challenge_abc123',
})
print(f'Status: {resp.status_code}')
print(f'Body:   {resp.text!r}')
assert resp.status_code == 200
assert resp.text == 'challenge_abc123'
print('\n✓ Webhook verificado correctamente')

Status: 200
Body:   'challenge_abc123'

✓ Webhook verificado correctamente


In [8]:
# B.2: Token incorrecto → 403
resp = client.get('/webhook', params={
    'hub.mode': 'subscribe',
    'hub.verify_token': 'wrong-token-xyz',
    'hub.challenge': 'challenge_abc123',
})
pprint(resp)
assert resp.status_code == 403
print('\n✓ Token incorrecto rechazado')

HTTP 403
{
  "status": "error",
  "message": "Verification failed"
}

✓ Token incorrecto rechazado


In [20]:
# B.3: Sin parámetros → 400
resp = client.get('/webhook')
pprint(resp)
assert resp.status_code == 400
print('\n✓ Parámetros faltantes rechazados')

HTTP 200
''


AssertionError: 

---
## C. Seguridad y Validación (POST /webhook)

Cada webhook de Meta lleva el header `X-Hub-Signature-256: sha256=<hmac>` calculado con el `APP_SECRET`. Sin firma válida → 403.

In [10]:
# C.1: Sin header de firma → 403
payload = make_text_msg('5215599990001', 'Test', 'Hola')
body = json.dumps(payload).encode('utf-8')
resp = client.post('/webhook', content=body, headers={'Content-Type': 'application/json'})
pprint(resp)
assert resp.status_code == 403
print('\n✓ Firma ausente rechazada')

HTTP 403
{
  "detail": "Invalid signature"
}

✓ Firma ausente rechazada


In [11]:
# C.2: Firma incorrecta → 403
payload = make_text_msg('5215599990001', 'Test', 'Hola')
resp = post_webhook(payload, bad_secret='wrong-secret-xyz')
pprint(resp)
assert resp.status_code == 403
print('\n✓ Firma inválida rechazada')

HTTP 403
{
  "detail": "Invalid signature"
}

✓ Firma inválida rechazada


In [12]:
# C.3: Status update (delivered/read) → 200, procesado silenciosamente
# WhatsApp envía 4 webhooks por mensaje: message, sent, delivered, read
for status in ['sent', 'delivered', 'read']:
    payload = make_status_update('5215512345678', status)
    resp = post_webhook(payload)
    print(f'  {status:10s} → HTTP {resp.status_code}')
    assert resp.status_code == 200
print('\n✓ Status updates ignorados correctamente (sin procesar como mensaje)')

  sent       → HTTP 200
  delivered  → HTTP 200
  read       → HTTP 200

✓ Status updates ignorados correctamente (sin procesar como mensaje)


In [14]:
# C.4: Payload válido en firma pero estructura inválida → 404
for label, payload in [
    ('sin entry', {'object': 'whatsapp_business_account'}),
    ('entry vacío', {'object': 'whatsapp_business_account', 'entry': []}),
    ('sin messages', {'object': 'whatsapp_business_account', 'entry': [{'changes': [{'value': {}}]}]}),
]:
    resp = post_webhook(payload)
    print(f'  {label:20s} → HTTP {resp.status_code}')
assert resp.status_code == 404
print('\n✓ Estructuras inválidas rechazadas con 404')

  sin entry            → HTTP 404
  entry vacío          → HTTP 404
  sin messages         → HTTP 404

✓ Estructuras inválidas rechazadas con 404


In [22]:
# C.5: Mensaje válido → 200 (procesamiento en background)
payload = make_text_msg('2', 'agustina', "hola")# 'dame dos mix de chocolate a la direccion Angeles 123, entrega a mi puerta a las 7pm')
resp = post_webhook(payload)
pprint(resp)
assert resp.status_code == 200
print('\n✓ Mensaje válido aceptado (background task lanzado)')


HTTP 200
{
  "status": "ok"
}

✓ Mensaje válido aceptado (background task lanzado)


In [78]:
print('Ahora, Bob, además del *Mix enchilado*, tengo la información de otros productos:\n\n1. *Mix Chocolate* - $200.00 MXN - 1.0 kg, que contiene:\n   - Almendra con chocolate\n   - Arándano con chocolate\n   - Uva pasa con chocolate\n   - Nuez natural\n   - Cacahuate natural\n   - Almendra natural\n\n2. *Nuez Natural Corazon* - $390.00 MXN - 900.0 gr, que es nuez pecanera de temporada.\n\nSi alguno de estos te interesa, házmelo saber!')

Ahora, Bob, además del *Mix enchilado*, tengo la información de otros productos:

1. *Mix Chocolate* - $200.00 MXN - 1.0 kg, que contiene:
   - Almendra con chocolate
   - Arándano con chocolate
   - Uva pasa con chocolate
   - Nuez natural
   - Cacahuate natural
   - Almendra natural

2. *Nuez Natural Corazon* - $390.00 MXN - 900.0 gr, que es nuez pecanera de temporada.

Si alguno de estos te interesa, házmelo saber!


---
## D. Unit Tests — Sin Servidor

Importa los módulos directamente y testea funciones aisladas. **El servidor no necesita estar corriendo.** Solo requiere las variables de entorno del Setup.

In [23]:
import sys

# El notebook vive en app/services/chatbot/
chatbot_dir = os.getcwd()
db_dir = os.path.normpath(os.path.join(chatbot_dir, '..', 'database'))

for p in [chatbot_dir, db_dir]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'chatbot dir: {chatbot_dir}')
print(f'schema dir:  {db_dir}')

chatbot dir: /home/sade/Documents/repos/toilav-bot/app/services/chatbot
schema dir:  /home/sade/Documents/repos/toilav-bot/app/services/database


In [24]:
from chatbot_schema import ConversationPhase
from whatsapp_utils import (
    Contact,
    UserMessageBuffer,
    WhatsappMessage,
    extract_message,
    is_valid_whatsapp_message,
    parse_text_for_whatsapp,
)

print('✓ Módulos importados sin servidor')

✓ Módulos importados sin servidor


In [8]:
# D.2: extract_message — casos soportados

# Texto
payload = make_text_msg('5215512345678', 'Juan', 'Hola, quiero información')
msg = extract_message(payload)
assert msg is not None
assert msg.contact.wa_id == '5215512345678'
assert msg.contact.name == 'Juan'
assert msg.text == 'Hola, quiero información'
assert msg.type == 'text'
print(f'texto  → wa_id={msg.contact.wa_id}, text={msg.text!r}')

# Imagen sin caption: text queda vacío en WhatsappMessage (lo maneja _extract_message_text)
payload = make_image_msg('5215500001111', 'Ana')
msg = extract_message(payload)
assert msg is not None and msg.type == 'image' and msg.text == ''
print(f'imagen → type={msg.type!r}, text={msg.text!r} (vacío; caption lo procesa el buffer)')

# Imagen con caption
payload = make_image_msg('5215500001111', 'Ana', caption='Mi comprobante de pago')
msg = extract_message(payload)
assert msg is not None
print(f'imagen+caption → type={msg.type!r}, image.caption preservado en payload original')

print('\n✓ extract_message OK')

NameError: name 'extract_message' is not defined

In [33]:
# D.3: parse_text_for_whatsapp — conversión de formato LLM → WhatsApp
cases = [
    ('**texto bold**',         '*texto bold*'),
    ('Ref 【doc001】 aquí',     'Ref  aquí'),
    ('*ya está bold*',          '*ya está bold*'),
    ('_cursiva_',               '_cursiva_'),
    ('**A** y **B**',           '*A* y *B*'),
]
for inp, expected in cases:
    result = parse_text_for_whatsapp(inp).strip()
    status = '✓' if result == expected else '✗'
    print(f'  {status}  {inp!r:30s} → {result!r}')

  ✓  '**texto bold**'               → '*texto bold*'
  ✓  'Ref 【doc001】 aquí'            → 'Ref  aquí'
  ✓  '*ya está bold*'               → '*ya está bold*'
  ✓  '_cursiva_'                    → '_cursiva_'
  ✓  '**A** y **B**'                → '*A* y *B*'


In [27]:
# D.4: is_valid_whatsapp_message
cases = [
    (make_text_msg('123', 'Test', 'Hola'), True,  'mensaje válido'),
    ({'object': 'whatsapp_business_account'}, False, 'sin entry'),
    ({}, False, 'vacío'),
    ({'object': 'x', 'entry': [{'changes': [{'value': {}}]}]}, False, 'sin messages'),
]
for payload, expected, label in cases:
    result = is_valid_whatsapp_message(payload)
    status = '✓' if result == expected else '✗'
    print(f'  {status}  {label:25s} → {result}')
print('\n✓ Validación de estructura OK')

  ✓  mensaje válido            → True
  ✓  sin entry                 → False
  ✓  vacío                     → False
  ✓  sin messages              → False

✓ Validación de estructura OK


In [28]:
# D.5: UserMessageBuffer — deduplicación
buf = UserMessageBuffer(max_seen=5)

def _msg(wamid: str, text: str = 'test') -> WhatsappMessage:
    return WhatsappMessage(id=wamid, contact=Contact('123', 'Test'), timestamp=0, text=text, type='text')

assert buf.is_duplicate(_msg('wamid.001')) is False  # primera vez → no dup
assert buf.is_duplicate(_msg('wamid.001')) is True   # retry   → dup
assert buf.is_duplicate(_msg('wamid.002')) is False  # nuevo   → no dup
print('✓ Deduplicación básica OK')

# LRU eviction con max_seen=3
buf2 = UserMessageBuffer(max_seen=3)
for i in range(5):
    buf2.is_duplicate(_msg(f'wamid.{i:03d}'))

# wamid.000 y wamid.001 deben haber sido evictados
assert buf2.is_duplicate(_msg('wamid.000')) is False, 'wamid.000 debería haber sido evictado'
assert buf2.is_duplicate(_msg('wamid.004')) is True,  'wamid.004 debería seguir en caché'
print('✓ LRU eviction OK (IDs viejos se olvidan, recientes se recuerdan)')

✓ Deduplicación básica OK
✓ LRU eviction OK (IDs viejos se olvidan, recientes se recuerdan)


In [29]:
# D.6: UserMessageBuffer — flush con ordenamiento por timestamp
buf = UserMessageBuffer()

# Mensajes agregados FUERA de orden (c antes que b)
for wamid, ts, text in [('a', 1000, 'Primero'), ('c', 1002, 'Tercero'), ('b', 1001, 'Segundo')]:
    msg = WhatsappMessage(id=wamid, contact=Contact('123', 'Test'), timestamp=ts, text=text, type='text')
    buf.add_message(msg)

combined = buf.flush()
lines = combined.split('\n')
print(f'Flush result: {lines}')
assert lines == ['Primero', 'Segundo', 'Tercero'], f'Orden incorrecto: {lines}'
print('\n✓ Flush ordena por timestamp correctamente')

Flush result: ['Primero', 'Segundo', 'Tercero']

✓ Flush ordena por timestamp correctamente


In [30]:
# D.7: ConversationPhase — todos los valores del enum
print('ConversationPhase:')
for phase in ConversationPhase:
    print(f'  {phase.name:25s} = {phase.value!r}')

ConversationPhase:
  GREETING                  = 'greeting'
  QA_LOOP                   = 'qa_loop'
  ORDER_BUILDING            = 'order_building'
  COLLECTING_DETAILS        = 'collecting_details'
  PENDING_APPROVAL          = 'pending_approval'
  PENDING_PAYMENT           = 'pending_payment'
  PENDING_DELIVERY          = 'pending_delivery'
  DELIVERY_IN_COURSE        = 'delivery_in_course'
  COMPLETED                 = 'completed'


---
## E. Flujo de Cliente vía HTTP

Sendea payloads completos al servidor. Revisa los **logs del servidor** para ver las respuestas del LLM.

Esquema de fases:
```
GREETING → QA_LOOP → ORDER_BUILDING → COLLECTING_DETAILS
         → PENDING_APPROVAL → PENDING_PAYMENT → PENDING_DELIVERY
         → DELIVERY_IN_COURSE → COMPLETED
```

In [29]:
# E.1: Fase 1 — Saludo (primer contacto, cliente nuevo)
# Bot debe presentarse, mencionar qué vende la tienda y preguntar en qué puede ayudar.
send_customer('5215599991001', 'María García', 'Hola, buenos días')

[María García] 'Hola, buenos días'                                           → HTTP 404


<Response [404 Not Found]>

In [32]:
# E.2: Fase 2 — Q&A: preguntas de productos
# Bot debe llamar a search_products antes de responder. Nunca inventar precios.
for text in [
    '¿Cuánto cuestan las almendras tostadas?',
    '¿Tienen nueces de la india? ¿Y pistaches?',
    '¿Cuáles son los métodos de pago?',
    '¿Hacen entregas a domicilio? ¿A dónde?',
]:
    send_customer('5215599991001', 'María García', text, wait=2.0) 

[María García] '¿Cuánto cuestan las almendras tostadas?'                     → HTTP 200
[María García] '¿Tienen nueces de la india? ¿Y pistaches?'                   → HTTP 200
[María García] '¿Cuáles son los métodos de pago?'                            → HTTP 200
[María García] '¿Hacen entregas a domicilio? ¿A dónde?'                      → HTTP 200


In [ ]:
# E.3: Fase 2 — Q&A: término ambiguo
# Bot debe pedir aclaración antes de buscar (según mega-prompt)
send_customer('5215599991001', 'María García', '¿Tienes de castilla?')
# Responder a la aclaración del bot:
send_customer('5215599991001', 'María García', 'Sí, nueces de Castilla')

In [ ]:
# E.4: Fase 2 — Knowledge gap: pregunta que el bot no puede responder
# Bot debe escalar con notify_owner y decirle al cliente que lo verifica
send_customer('5215599991001', 'María García', '¿Hacen descuentos para compras al mayoreo? Más de 2kg')

In [ ]:
# E.5: Fase 3 — Construcción de pedido
# Bot debe: verificar precios (search_products), crear orden (create_order)
for text in [
    'Quiero pedir almendras tostadas de 500g y pistaches de 100g',
    'También agrega arándanos deshidratados, 100 gramos',
    '¿Puedes mostrarme el resumen del pedido?',
]:
    send_customer('5215599992001', 'Carlos Ramírez', text)

In [ ]:
# E.6: Fase 3 — Modificaciones al pedido
for text in [
    'Quita los arándanos',
    'Cámbia las almendras a 200g en lugar de 500g',
    'Agrega nueces de la india, 200 gramos',
]:
    send_customer('5215599992001', 'Carlos Ramírez', text)

In [ ]:
# E.7: Fase 3 → 4 — Confirmar pedido
# Bot debe notificar al dueño con notify_owner y pedir datos de entrega
send_customer('5215599992001', 'Carlos Ramírez', 'Perfecto, así está bien. Confirmo el pedido.')

In [ ]:
# E.8: Fase 4 — Recolección de datos de entrega
# Bot pide nombre, dirección, instrucciones, ventana horaria, método de pago
send_customer(
    '5215599992001', 'Carlos Ramírez',
    'Me llamo Carlos Ramírez. Mi dirección es Calle 15 #45-23, Col. Centro. '
    'Quiero que entreguen mañana entre 2 y 4 pm. Pago por transferencia bancaria.'
)

In [ ]:
# E.9: Imagen CON caption — simula comprobante de pago
# Bot debe tratar el caption como el texto del mensaje
payload = make_image_msg(
    '5215599992001', 'Carlos Ramírez',
    caption='Aquí te mando mi comprobante de transferencia por $205'
)
resp = post_webhook(payload)
print(f'Imagen con caption → HTTP {resp.status_code}')
time.sleep(5)

In [ ]:
# E.10: Tipos de mensaje no soportados
# Respuestas hardcodeadas en _UNSUPPORTED_TYPE_RESPONSE
for msg_type in ['audio', 'sticker', 'location', 'document']:
    payload = make_typed_msg('5215599993001', 'Ana López', msg_type)
    resp = post_webhook(payload)
    print(f'  {msg_type:10s} → HTTP {resp.status_code}')
    time.sleep(5)

print('\nRevisa los logs para ver los mensajes de aviso del bot')

In [ ]:
# E.11: Reacción — debe ignorarse silenciosamente (reaction → None en _UNSUPPORTED_TYPE_RESPONSE)
reaction_payload = {
    'object': 'whatsapp_business_account',
    'entry': [{'changes': [{'value': {
        'contacts': [{'profile': {'name': 'Ana'}, 'wa_id': '5215599993001'}],
        'messages': [{
            'from': '5215599993001',
            'id': next_wamid(),
            'timestamp': str(int(time.time())),
            'type': 'reaction',
            'reaction': {'message_id': 'wamid.prev001', 'emoji': '❤️'},
        }],
    }}]}],
}
resp = post_webhook(reaction_payload)
print(f'Reaction → HTTP {resp.status_code}')
time.sleep(5)
print('No debería aparecer respuesta del bot en los logs (silenciosamente ignorado)')

In [ ]:
# E.12: Mensaje interactivo — respuesta a botón
# El título del botón se extrae como texto del mensaje
payload = make_interactive_msg('5215599991001', 'María García', 'Sí, quiero hacer un pedido', 'btn_order')
resp = post_webhook(payload)
print(f'Interactive button → HTTP {resp.status_code}')
time.sleep(5)

---
## F. Comandos del Dueño

El dueño envía comandos slash desde su número personal (`OWNER_WA_ID`). Todos los mensajes del dueño van a `handle_owner_command`.

> **Bug conocido (actual):** `handle_owner_command` recibe un objeto `WhatsappMessage` pero su firma dice `text: str`. El `text.startswith('/')` falla con `AttributeError`, que queda atrapado en el callback del task y se loguea como error. Los handlers por fase están marcados como TODO. Estos tests documentan el comportamiento actual y servirán para verificar el fix cuando se implemente.

In [ ]:
# F.1: Comandos básicos de aprobación de órdenes
for cmd in ['/approve', '/ok']:    # aliases
    send_owner(cmd, wait=1)

In [ ]:
# F.2: Rechazo con motivo
send_owner('/reject No tenemos stock de almendras esta semana')
send_owner('/no Fuera de zona de entrega')

In [ ]:
# F.3: Respuesta a knowledge gap
send_owner('/answer Sí, hacemos 10% de descuento en pedidos mayores a $500. Escríbenos.')
send_owner('/a Tenemos envíos los sábados hasta las 2pm')  # alias /a

In [ ]:
# F.4: Takeover / resume
send_owner('/intervene')    # Bot se silencia, dueño toma control
time.sleep(1)
send_owner('/resume')       # Bot retoma
time.sleep(1)
send_owner('/back')         # alias
time.sleep(1)
send_owner('/continue-bot') # alias

In [ ]:
# F.5: Confirmación de pago y entrega
for cmd in [
    '/payment-received',
    '/paid',                                          # alias
    '/payment-rejected El monto estaba incompleto',
    '/on-the-way',
    '/delivery',                                      # alias
    '/delivered',
    '/done',                                          # alias
    '/delivery-issue Se cayó el paquete en la ruta',
]:
    send_owner(cmd, wait=1)

In [ ]:
# F.6: Cambio de dirección de entrega
send_owner('/address-ok')
send_owner('/address-rejected No hacemos entregas en ese municipio')

In [ ]:
# F.7: Comandos de utilidad
for cmd in ['/pending', '/status Carlos Ramírez', '/status 5215599992001', '/help']:
    send_owner(cmd, wait=1)

In [ ]:
# F.8: Texto libre del dueño (sin /intervene activo)
# Debe ignorarse (no hay HUMAN_TAKEOVER activo)
payload = make_text_msg(OWNER_WA_ID, 'Owner', 'Hola sistema, ¿cómo estás?')
resp = post_webhook(payload)
print(f'Free-text del dueño → HTTP {resp.status_code}')
time.sleep(3)
print('El texto libre del dueño debe loguearse y descartarse sin respuesta')

---
## G. Deduplicación de Mensajes

WhatsApp puede reenviar el mismo webhook varias veces (reintentos de red). El `UserMessageBuffer` usa un LRU cache por `wamid` para procesar cada mensaje solo una vez.

In [ ]:
# G.1: Mismo wamid dos veces — el segundo debe ignorarse
fixed_id = 'wamid.dedup_001'
wa_id    = '5215599994001'

payload = make_text_msg(wa_id, 'Pedro', '¿Tienen almendras?', msg_id=fixed_id)

r1 = post_webhook(payload)
time.sleep(0.3)   # Simula retry de red sin esperar el debounce
r2 = post_webhook(payload)  # Mismo wamid

print(f'Primer envío : HTTP {r1.status_code}')
print(f'Reintento    : HTTP {r2.status_code}')
print()
time.sleep(5)  # Espera que el primer mensaje se procese
print('Ambos devuelven 200 (siempre ACK a WhatsApp).')
print('Pero solo el primero genera una llamada al LLM.')
print('Busca en logs: "Duplicate message wamid.dedup_001 ... skipping"')

In [ ]:
# G.2: 5 reintentos del mismo mensaje (red inestable)
fixed_id = 'wamid.dedup_002'
wa_id    = '5215599994002'

payload = make_text_msg(wa_id, 'Rosa', '¿Cuánto cuesta el mix?', msg_id=fixed_id)

for i in range(5):
    r = post_webhook(payload)
    print(f'  Intento {i+1}: HTTP {r.status_code}')
    time.sleep(0.1)

time.sleep(5)
print('\nSolo el intento #1 debería aparecer en el log del LLM')

---
## H. Batching por Debounce

Clientes que envían varios mensajes rápidos (stream of consciousness). El `UserMessageBuffer` acumula mensajes durante **3 segundos** de inactividad y luego los pasa al LLM como un único mensaje combinado, ordenado por timestamp.

In [ ]:
# H.1: Burst de 3 mensajes dentro de la ventana de debounce
# Deben llegar al LLM como un solo mensaje combinado
wa_id     = '5215599995001'
name      = 'Luis Torres'
base_ts   = int(time.time())

messages = [
    (0, 'Hola, tengo unas preguntas...'),
    (1, '¿Cuánto cuestan las almendras?'),
    (2, '¿Y tienen envío a domicilio?'),
]

for ts_offset, text in messages:
    payload = make_text_msg(wa_id, name, text, timestamp=base_ts + ts_offset)
    resp = post_webhook(payload)
    print(f'  Enviado: {text!r:50s} → HTTP {resp.status_code}')
    time.sleep(0.4)   # Dentro de la ventana de 3 segundos

print('\nEsperando que cierre la ventana de debounce (4 segundos)...')
time.sleep(4)
print('Busca en logs: UN solo "Processing buffered messages" con los 3 mensajes combinados')

In [ ]:
# H.2: Dos bursts separados → dos llamadas LLM distintas
wa_id = '5215599995002'
name  = 'Elena Ruiz'

print('=== Burst 1 ===')
for text in ['¿Qué tienen disponible?', '¿Y precios?']:
    payload = make_text_msg(wa_id, name, text)
    post_webhook(payload)
    print(f'  → {text!r}')
    time.sleep(0.5)

print('Esperando cierre del burst 1 (5s)...')
time.sleep(5)

print('\n=== Burst 2 ===')
for text in ['Quiero hacer un pedido', 'De almendras tostadas, 200 gramos']:
    payload = make_text_msg(wa_id, name, text)
    post_webhook(payload)
    print(f'  → {text!r}')
    time.sleep(0.5)

time.sleep(5)
print('\nDeben aparecer DOS llamadas LLM separadas en los logs')

---
## I. Usuarios Concurrentes

Cada cliente tiene su propio `UserMessageBuffer`. Los mensajes de diferentes usuarios no deben interferir entre sí.

In [ ]:
async def _send_async(wa_id: str, name: str, text: str) -> tuple[str, int]:
    payload = make_text_msg(wa_id, name, text)
    body = json.dumps(payload, ensure_ascii=False).encode('utf-8')
    sig  = sign(body)
    async with httpx.AsyncClient(base_url=BASE_URL, timeout=30) as ac:
        resp = await ac.post(
            '/webhook', content=body,
            headers={'Content-Type': 'application/json', 'X-Hub-Signature-256': sig},
        )
    return name, resp.status_code

clientes = [
    ('5215599996001', 'Sofía',      '¿Qué productos tienen disponibles?'),
    ('5215599996002', 'Miguel',     '¿Cuánto cuestan las nueces de la india?'),
    ('5215599996003', 'Isabel',     'Quiero hacer un pedido de almendras tostadas'),
    ('5215599996004', 'Javier',     '¿Hacen entregas los sábados?'),
    ('5215599996005', 'Valentina',  '¿Tienen alguna promoción activa?'),
]

results = await asyncio.gather(*[_send_async(wa_id, name, text) for wa_id, name, text in clientes])
for name, status in results:
    print(f'  {name:12s} → HTTP {status}')

print('\nRevisa los logs: 5 buffers independientes, 5 LLM calls (o batching por usuario)')

---
## J. Edge Cases

In [ ]:
# J.1: Texto vacío
payload = make_text_msg('5215599997001', 'Test', '')
resp = post_webhook(payload)
pprint(resp)
time.sleep(5)

HTTP 404
{
  "detail": "Not Found"
}


: 

In [ ]:
# J.2: Mensaje muy largo (stress test contexto del LLM)
long = '¿Tienen ' + 'almendras, nueces, pistaches, arándanos, castañas, ' * 15 + 'disponibles?'
print(f'Longitud: {len(long)} caracteres')
send_customer('5215599997002', 'Stress Test', long)

In [ ]:
# J.3: Mensaje en inglés — bot debe responder en español (regla del mega-prompt)
send_customer('5215599997003', 'John', 'Hello! Do you have almonds? How much do they cost?')

In [ ]:
# J.4: Pregunta fuera de contexto — bot debe redirigir amablemente
for msg in [
    '¿Me ayudas con mis tareas de matemáticas?',
    '¿Cuál es la capital de Francia?',
    '¿Puedes escribirme un poema?',
]:
    send_customer('5215599997004', 'Curioso', msg)

In [ ]:
# J.5: Comando de dueño enviado desde número de cliente (no-OWNER_WA_ID)
# Debe procesarse como mensaje regular al LLM, no como comando
payload = make_text_msg('5215500000999', 'NotOwner', '/approve')
resp = post_webhook(payload)
pprint(resp)
time.sleep(5)
print('/approve desde no-dueño → debe ir al LLM como texto normal')

In [ ]:
# J.6: Caracteres especiales, emojis y acentos
send_customer(
    '5215599997005', 'María',
    '¡Hola! 😊 ¿Tienen las almendras más ricas de todo México? 🇲🇽 '
    'Quiero algo *especial* para regalo...'
)

---
## K. Flujo E2E Completo

Simula una compra completa de principio a fin:
`GREETING → QA_LOOP → ORDER_BUILDING → COLLECTING_DETAILS → PENDING_APPROVAL → PENDING_PAYMENT → PENDING_DELIVERY → DELIVERY_IN_COURSE → COMPLETED`

In [ ]:
E2E_WA   = '5215599998001'
E2E_NAME = 'Fernanda Morales'

print('=' * 70)
print(f'E2E — {E2E_NAME}')
print('=' * 70)

customer_steps = [
    ('Saludo inicial',       'Hola, buenos días'),
    ('Pregunta catálogo',    '¿Qué tipos de nueces tienen disponibles?'),
    ('Consulta de precio',   '¿Cuánto cuestan las almendras tostadas de 500g?'),
    ('Zona de entrega',      '¿Hacen entregas en el centro histórico de Guanajuato?'),
    ('Iniciar pedido',       'Quiero pedir almendras tostadas de 500g y pistaches de 100g'),
    ('Ver resumen',          '¿Puedes mostrarme el resumen de mi pedido?'),
    ('Confirmar pedido',     'Perfecto, confirmo el pedido. Así está bien.'),
    ('Datos de entrega',     'Me llamo Fernanda Morales. Dirección: Av. Juárez 45, Col. Centro. '
                             'Entrega mañana entre 10am y 1pm. Pago por transferencia.'),
    ('Status pre-aprobación', '¿Ya revisaron mi pedido?'),
]

for phase, msg in customer_steps:
    print(f'\n[{phase}]')
    send_customer(E2E_WA, E2E_NAME, msg)

In [ ]:
print('=== LADO DUEÑO ===')

# Aprobación del pedido
print('\n[Owner: aprueba pedido]')
send_owner('/approve', wait=3)

# Cliente consulta status
print('\n[Cliente: confirma aprobación]')
send_customer(E2E_WA, E2E_NAME, '¿Ya fue aprobado mi pedido?')

# Cliente envía comprobante de pago (imagen con caption)
print('\n[Cliente: envía comprobante de pago]')
payload = make_image_msg(E2E_WA, E2E_NAME, caption='Aquí mi comprobante de transferencia por $255 💳')
post_webhook(payload)
time.sleep(5)

# Dueño confirma pago
print('\n[Owner: confirma pago recibido]')
send_owner('/payment-received', wait=3)

# Dueño envía a domicilio
print('\n[Owner: sale a entregar]')
send_owner('/on-the-way', wait=3)

# Cliente consulta estado durante entrega
print('\n[Cliente: status durante entrega]')
send_customer(E2E_WA, E2E_NAME, '¿Ya van en camino?')

# Dueño confirma entrega
print('\n[Owner: confirma entrega]')
send_owner('/delivered', wait=3)

print('\n' + '=' * 70)
print('E2E COMPLETO')

---
## L. Tests Directos del Agente LLM

Llama a `agent_generate_response` directamente, sin HTTP ni debounce. Ideal para iterar rápido sobre el comportamiento del prompt y las herramientas.

**Requiere:** `OPENAI_API_KEY` real en el entorno (celda 0).

In [ ]:
from yalti import StoreInfo, agent_generate_response, _pending_orders

STORE = StoreInfo(
    name='Tremenda Nuez',
    description=(
        'Tienda de nueces y frutos secos online desde WhatsApp. '
        'No tenemos tienda física. Entregamos directamente a domicilio en Guanajuato.'
    ),
    properties={},
)

print(f'✓ Agente importado')
print(f'  Modelo:  openai:gpt-4o-mini')
print(f'  Tienda:  {STORE.name}')

In [ ]:
# L.1: Turno único — saludo inicial
response, history = await agent_generate_response(
    message='Hola, buenos días',
    wa_id='direct_001',
    name='Fernanda',
    store=STORE,
    history=[],
)
print(f'Bot: {response}')
print(f'\nHistorial: {len(history)} mensajes')

In [ ]:
# L.2: Conversación multi-turno — Q&A con herramientas
wa_id   = 'direct_002'
history = []

for user_msg in [
    'Hola, ¿qué productos tienen?',
    '¿Cuánto cuestan las almendras de 500g?',
    '¿Y los pistaches cuánto?',
    '¿Hacen entregas a domicilio?',
]:
    print(f'\nUsuario: {user_msg}')
    response, history = await agent_generate_response(
        message=user_msg, wa_id=wa_id, name='Carlos', store=STORE, history=history
    )
    print(f'Bot:     {response}')

print(f'\nHistorial acumulado: {len(history)} mensajes')

In [ ]:
# L.3: Construcción de pedido — crea, modifica y confirma
wa_id   = 'direct_003'
history = []

for user_msg in [
    'Quiero almendras tostadas de 500g y nueces de la india de 200g',
    'Quita las nueces de la india, mejor agrega arándanos 100g',
    'Cámbia las almendras a 200g',
    '¿Puedes mostrarme el resumen del pedido?',
    'Todo bien, confirmo el pedido. Me llamo Sara Gómez.',
]:
    print(f'\nUsuario: {user_msg}')
    response, history = await agent_generate_response(
        message=user_msg, wa_id=wa_id, name='Sara Gómez', store=STORE, history=history
    )
    print(f'Bot:     {response}')

# Inspecciona el pedido en memoria
if wa_id in _pending_orders:
    order = _pending_orders[wa_id]
    print(f'\n--- Pedido en memoria ---')
    print(f'ID: {order["id"]}  |  Cliente: {order["customer"]}')
    for item in order['items']:
        print(f'  • {item["qty"]}x {item["name"]:30s}  ${item["unit_price"] * item["qty"]:.0f}')
    print(f'  Total: ${order["total"]:.0f}')
else:
    print(f'\nNo se encontró pedido para {wa_id}')

In [ ]:
# L.4: Knowledge gap — bot debe escalar con notify_owner
# La llamada al WhatsApp API fallará (token de prueba) pero el bot debe responder correctamente
print('Pregunta que el bot no puede resolver con search_products...')
response, _ = await agent_generate_response(
    message='¿Hacen descuentos para eventos corporativos? Necesito 50 bolsas para regalo.',
    wa_id='direct_004',
    name='Diego',
    store=STORE,
    history=[],
)
print(f'Bot: {response}')
print('\nSi llamó a notify_owner, habrá un error de red en los logs (token inválido).')
print('El bot debe decirle al cliente que lo está verificando con el equipo.')

In [ ]:
# L.5: Redireccionamiento de temas fuera de contexto
for question in [
    '¿Me ayudas con una receta de cocina?',
    '¿Cuál es la capital de Italia?',
    'Escríbeme un poema sobre el otoño',
]:
    response, _ = await agent_generate_response(
        message=question, wa_id='direct_005', name='Curioso', store=STORE, history=[]
    )
    print(f'Q: {question!r}')
    print(f'A: {response[:120]}...' if len(response) > 120 else f'A: {response}')
    print()

In [ ]:
# L.6: Memoria de contexto — referencias a mensajes anteriores
wa_id   = 'direct_006'
history = []

print('Turno 1: establece contexto')
response, history = await agent_generate_response(
    'Busco algo para regalarle a mi mamá que le encantan los frutos secos',
    wa_id, 'Gabriela', STORE, history
)
print(f'Bot: {response}\n')

print('Turno 2: referencia implícita al contexto anterior')
response, history = await agent_generate_response(
    '¿Cuál me recomiendas para el regalo que te mencioné?',
    wa_id, 'Gabriela', STORE, history
)
print(f'Bot: {response}\n')

print('Turno 3: pregunta implícita de precio')
response, history = await agent_generate_response(
    '¿Cuánto me costaría eso?',
    wa_id, 'Gabriela', STORE, history
)
print(f'Bot: {response}')
print(f'\nHistorial total: {len(history)} mensajes')

In [ ]:
# L.7: Término ambiguo → bot debe pedir aclaración antes de buscar (regla del mega-prompt)
wa_id   = 'direct_007'
history = []

print('Término ambiguo: "castilla"')
response, history = await agent_generate_response(
    '¿Tienes de castilla?', wa_id, 'Test', STORE, []
)
print(f'Bot: {response}\n')

print('Aclaración del cliente:')
response, history = await agent_generate_response(
    'Sí, nueces de Castilla, las grandotas', wa_id, 'Test', STORE, history
)
print(f'Bot: {response}\n')

print('Pregunta clara (no debe pedir aclaración):')
response, _ = await agent_generate_response(
    '¿Cuánto cuestan las almendras tostadas de 500 gramos?', 'direct_007b', 'Test', STORE, []
)
print(f'Bot: {response}')